# Notebook 02 — Constraint Sampling

This notebook compares two access models for the mod30 residue manifold:

1. **Uniform sampling** over candidate integers.
2. **Constraint sampling** restricted to valid prime lanes in \(\mathbb{Z}/30\mathbb{Z}\).

The goal is to show that structure-aligned access changes signal efficiency before any learning model is applied.

## 1. Setup

Figures are saved as SVG only. Notebook 01 outputs are loaded when available; otherwise this notebook rebuilds the minimal mod30 tables so it can still run independently.

In [ ]:
# Notebook 02 — Constraint Sampling

import os
from math import gcd

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Vector-friendly SVG defaults
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["figure.dpi"] = 120

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

MODULUS = 30
N_MAX = 10_000
VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]


def save_svg(fig, name):
    """Save a matplotlib figure as one canonical SVG file."""
    path = f"figures/{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")


def simple_primes_upto(n):
    """Return primes <= n using a small sieve; no external dependencies."""
    if n < 2:
        return []
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    for p in range(2, int(np.sqrt(n)) + 1):
        if sieve[p]:
            sieve[p*p:n+1:p] = False
    return np.flatnonzero(sieve).tolist()

## 2. Load or rebuild Notebook 01 data

Notebook 02 uses the same canonical valid-lane set from Notebook 01:

\[
\{1,7,11,13,17,19,23,29\} \subset \mathbb{Z}/30\mathbb{Z}.
\]

In [ ]:
residue_path = "data/residues_mod30.csv"
prime_path = "data/primes_mod30.csv"
summary_path = "data/residue_lane_summary_mod30.csv"

if os.path.exists(residue_path) and os.path.exists(prime_path):
    df_res = pd.read_csv(residue_path)
    df_primes = pd.read_csv(prime_path)
    print("Loaded Notebook 01 data from data/.")
else:
    print("Notebook 01 data not found; rebuilding minimal mod30 tables.")
    numbers = np.arange(0, N_MAX + 1)
    df_res = pd.DataFrame({
        "n": numbers,
        "residue_mod30": numbers % MODULUS,
        "is_valid_lane": np.isin(numbers % MODULUS, VALID_LANES_MOD30),
    })

    primes = np.array(simple_primes_upto(N_MAX))
    primes_excluding_factors = primes[~np.isin(primes, [2, 3, 5])]
    df_primes = pd.DataFrame({
        "prime": primes_excluding_factors,
        "residue_mod30": primes_excluding_factors % MODULUS,
    })

    df_res.to_csv(residue_path, index=False)
    df_primes.to_csv(prime_path, index=False)

print("Valid lanes:", VALID_LANES_MOD30)
print("Candidate rows:", len(df_res))
print("Prime rows excluding 2, 3, 5:", len(df_primes))

## 3. Define sampling spaces

The comparison is intentionally simple:

- **Uniform access:** draw from all candidate integers \(2 \le n < N\).
- **Constrained access:** draw only from integers whose residue lies in a valid mod30 lane.

This separates *access to structure* from any downstream learning model.

In [ ]:
all_numbers = np.arange(2, N_MAX)
valid_numbers = all_numbers[np.isin(all_numbers % MODULUS, VALID_LANES_MOD30)]

print("All candidate numbers:", len(all_numbers))
print("Valid-lane candidate numbers:", len(valid_numbers))
print("Valid-lane density:", len(valid_numbers) / len(all_numbers))


def sample_uniform(rng, size):
    return rng.choice(all_numbers, size=size, replace=True)


def sample_constrained(rng, size):
    return rng.choice(valid_numbers, size=size, replace=True)


def signal_rate(samples):
    """Fraction of samples landing in valid mod30 lanes."""
    return float(np.mean(np.isin(samples % MODULUS, VALID_LANES_MOD30)))


def unique_valid_lanes_seen(samples):
    """Number of distinct valid lanes represented by samples."""
    residues = set((samples % MODULUS).tolist())
    return len(residues.intersection(VALID_LANES_MOD30))

## 4. Monte Carlo sampling experiment

For each sample size, repeated trials estimate two quantities:

- `signal_rate`: fraction of draws in valid lanes.
- `unique_lanes_seen`: number of valid lanes observed at least once.

In [ ]:
sample_sizes = [10, 25, 50, 100, 250, 500, 1000]
trials = 200
rng = np.random.default_rng(9423)

records = []

for size in sample_sizes:
    for trial in range(trials):
        u = sample_uniform(rng, size)
        c = sample_constrained(rng, size)

        records.append({
            "mode": "uniform",
            "sample_size": size,
            "trial": trial,
            "signal_rate": signal_rate(u),
            "unique_lanes_seen": unique_valid_lanes_seen(u),
        })

        records.append({
            "mode": "constrained",
            "sample_size": size,
            "trial": trial,
            "signal_rate": signal_rate(c),
            "unique_lanes_seen": unique_valid_lanes_seen(c),
        })

df_trials = pd.DataFrame(records)
df_trials.head()

## 5. Summary table

The constrained sampler should produce signal rate \(1.0\) by construction, while uniform sampling should stay near \(8/30\).

In [ ]:
summary = (
    df_trials
    .groupby(["mode", "sample_size"])
    .agg(
        signal_rate_mean=("signal_rate", "mean"),
        signal_rate_std=("signal_rate", "std"),
        lanes_seen_mean=("unique_lanes_seen", "mean"),
        lanes_seen_std=("unique_lanes_seen", "std"),
    )
    .reset_index()
)

summary

## 6. Figure — residue counts under two access models

This figure shows one sampled batch. Uniform access spends many samples outside valid lanes; constrained access places all samples on the residue manifold.

In [ ]:
example_size = 250
rng_example = np.random.default_rng(17)

u = sample_uniform(rng_example, example_size)
c = sample_constrained(rng_example, example_size)

uniform_counts = np.bincount(u % MODULUS, minlength=MODULUS)
constrained_counts = np.bincount(c % MODULUS, minlength=MODULUS)
count_matrix = np.vstack([uniform_counts, constrained_counts])

fig, ax = plt.subplots(figsize=(14, 2.8))
im = ax.imshow(count_matrix, aspect="auto")

ax.set_title("Sampling access models over residue classes mod 30")
ax.set_xlabel("Residue class r mod 30")
ax.set_yticks([0, 1])
ax.set_yticklabels(["uniform", "constrained"])
ax.set_xticks(np.arange(MODULUS))

# mark valid lanes with small labels above the axis
for r in VALID_LANES_MOD30:
    ax.text(r, -0.58, "✓", ha="center", va="center", fontsize=10)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("sample count")

save_svg(fig, "sampling_uniform_vs_constrained")
plt.show()

## 7. Figure — signal rate vs sample size

This is the core access-model result: constrained sampling changes the signal rate before any learning algorithm is introduced.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))

for mode in ["uniform", "constrained"]:
    sub = summary[summary["mode"] == mode]
    ax.errorbar(
        sub["sample_size"],
        sub["signal_rate_mean"],
        yerr=sub["signal_rate_std"],
        marker="o",
        capsize=3,
        label=mode,
    )

ax.axhline(len(VALID_LANES_MOD30) / MODULUS, linestyle="--", linewidth=1, label="8/30 baseline")
ax.set_xscale("log")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Signal rate under uniform vs constrained sampling")
ax.set_xlabel("sample size")
ax.set_ylabel("fraction in valid lanes")
ax.legend()

save_svg(fig, "sampling_signal_rate")
plt.show()

## 8. Figure — valid-lane recovery curve

Lane recovery measures how quickly each access model observes all eight valid residue lanes.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))

for mode in ["uniform", "constrained"]:
    sub = summary[summary["mode"] == mode]
    ax.errorbar(
        sub["sample_size"],
        sub["lanes_seen_mean"],
        yerr=sub["lanes_seen_std"],
        marker="o",
        capsize=3,
        label=mode,
    )

ax.axhline(len(VALID_LANES_MOD30), linestyle="--", linewidth=1, label="all 8 valid lanes")
ax.set_xscale("log")
ax.set_ylim(0, 8.5)
ax.set_title("Valid-lane recovery under two sampling models")
ax.set_xlabel("sample size")
ax.set_ylabel("unique valid lanes seen")
ax.legend()

save_svg(fig, "sampling_recovery_curve")
plt.show()

## 9. Save data outputs

These CSVs become inputs for later notebooks and paper figures.

In [ ]:
df_trials.to_csv("data/constraint_sampling_trials.csv", index=False)
summary.to_csv("data/constraint_sampling_summary.csv", index=False)

print("Saved: data/constraint_sampling_trials.csv")
print("Saved: data/constraint_sampling_summary.csv")

## 10. Interpretation

Uniform sampling treats the ambient integer space as the access model. Constraint sampling treats the valid residue lanes as the access model. The difference is not a learning result yet; it is a readout/access result.

**Paper claim:** Uniform sampling spends most evaluations outside valid residue lanes, while constrained sampling places all evaluations inside the residue manifold. This changes access efficiency before any learning model is applied.

**Tang bridge:** Tang-style dequantization shows that speedup claims depend on matched access to structure; this notebook makes that access model explicit for mod30 residue manifolds.

## 11. Optional download bundle

Uncomment the final two lines to trigger a Colab browser download.

In [ ]:
# --- Optional: Download outputs (uncomment last lines to trigger) ---

import os
import zipfile

zip_name = "02_constraint_sampling_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)